In [11]:
# 02_load_procedures_to_clickhouse.ipynb

import pymssql
import pandas as pd
from clickhouse_driver import Client
from datetime import datetime
import os

print("=== Шаг 1: Подключение к MSSQL и выгрузка процедур ===\n")

# ============================================
# 1. Параметры подключения (как в твоём коде)
# ============================================
DB_HOST = "db22.vra.local"  # или db23
DB_USER = "sa"
DB_PASSWORD = "sasa"
DB_NAME = "mobile_Makeevka"  # или mobile_Bryansk

print(f"Подключение к MSSQL: {DB_HOST}/{DB_NAME}")

# ============================================
# 2. Функция подключения (как у тебя)
# ============================================
def get_mssql_connection():
    """Создаёт подключение к MSSQL"""
    return pymssql.connect(
        server=DB_HOST,
        user=DB_USER,
        password=DB_PASSWORD,
        database=DB_NAME,
        timeout=120,
        login_timeout=120,
        charset='UTF-8'
    )

# ============================================
# 3. Выгрузка списка процедур и их параметров
# ============================================
conn = get_mssql_connection()

# Запрос метаданных процедур
query_params = """
SELECT 
    SPECIFIC_SCHEMA as proc_schema,
    SPECIFIC_NAME as proc_name,
    PARAMETER_NAME as param_name,
    DATA_TYPE as param_type,
    ORDINAL_POSITION as param_order,
    CASE WHEN PARAMETER_MODE = 'OUT' THEN 1 ELSE 0 END as is_output
FROM INFORMATION_SCHEMA.PARAMETERS
WHERE SPECIFIC_SCHEMA = 'dbo'
ORDER BY SPECIFIC_NAME, ORDINAL_POSITION
"""

print("Выгружаем параметры процедур...")
df_params = pd.read_sql(query_params, conn)
print(f" Загружено {len(df_params)} записей (процедуры + параметры)")

# ============================================
# 4. Выгрузка полного текста процедур
# ============================================
# Получаем уникальные имена процедур
unique_procs = df_params['proc_name'].unique()
print(f"\nНайдено уникальных процедур: {len(unique_procs)}")

client.execute("TRUNCATE TABLE procedures_metadata.procedures_params")
client.execute("TRUNCATE TABLE procedures_metadata.procedures_full_text")
print(" Старые данные удалены")

print("\n--- Загрузка полных текстов всех процедур ---")

proc_definitions = []
total = len(unique_procs)

for i, proc_name in enumerate(unique_procs):
    try:
        cursor = conn.cursor()
        cursor.execute(f"EXEC sp_helptext '{proc_name}'")
        rows = cursor.fetchall()
        definition = '\n'.join([row[0] for row in rows if row[0]])
        
        proc_definitions.append({
            'proc_name': proc_name,
            'proc_definition': definition,
            'load_date': datetime.now()
        })
        cursor.close()
        
        if (i + 1) % 100 == 0:
            print(f"  Загружено {i + 1} из {total} процедур")
            
    except Exception as e:
        print(f"   Ошибка при загрузке {proc_name}: {e}")
        proc_definitions.append({
            'proc_name': proc_name,
            'proc_definition': '',
            'load_date': datetime.now()
        })

df_procs = pd.DataFrame(proc_definitions)
print(f" Загружено определений: {len(df_procs)}")

# ПРОВЕРКА: что данные есть
print(f"  Непустых определений: {(df_procs['proc_definition'] != '').sum()}")

conn.close()
print(" Подключение к MSSQL закрыто")

# ============================================
# 5. Подключение к ClickHouse и создание таблиц
# ============================================
print("\n=== Шаг 2: Сохранение в ClickHouse ===\n")

client = Client(host='my_clickhouse', port=9000, user='default', password='')

# Создаём базу данных (если нет)
client.execute("CREATE DATABASE IF NOT EXISTS procedures_metadata")

# Создаём таблицу для параметров процедур
client.execute("""
    CREATE TABLE IF NOT EXISTS procedures_metadata.procedures_params (
        proc_schema String,
        proc_name String,
        param_name String,
        param_type String,
        param_order Int32,
        is_output Int8,
        load_date DateTime DEFAULT now()
    ) ENGINE = MergeTree()
    ORDER BY (proc_name, param_order)
""")

# Создаём таблицу для полных текстов процедур
client.execute("""
    CREATE TABLE IF NOT EXISTS procedures_metadata.procedures_full_text (
        proc_name String,
        proc_definition String,
        load_date DateTime DEFAULT now()
    ) ENGINE = MergeTree()
    ORDER BY proc_name
""")

print(" Таблицы в ClickHouse созданы")

# 6. Загрузка данных в ClickHouse (исправлено)
# ============================================

print("\nЗагружаем полные тексты в ClickHouse...")

# Преобразуем в список кортежей
data_procs = []
for _, row in df_procs.iterrows():
    data_procs.append((
        row['proc_name'],
        row['proc_definition'],
        row['load_date']
    ))

# Вставляем
client.execute(
    "INSERT INTO procedures_metadata.procedures_full_text (proc_name, proc_definition, load_date) VALUES",
    data_procs
)
print(f" Загружено {len(data_procs)} записей в procedures_full_text")

# Проверяем
result = client.execute("SELECT COUNT(*) FROM procedures_metadata.procedures_full_text")
print(f"  Записей в ClickHouse: {result[0][0]}")

# ============================================
# Загрузка параметров в ClickHouse
# ============================================
print("\nЗагружаем параметры процедур...")

# Добавляем дату загрузки
df_params['load_date'] = datetime.now()

# Преобразуем в список кортежей
data_params = []
for _, row in df_params.iterrows():
    data_params.append((
        row['proc_schema'],
        row['proc_name'],
        row['param_name'] if row['param_name'] else '',
        row['param_type'],
        row['param_order'],
        row['is_output'],
        row['load_date']
    ))

client.execute(
    "INSERT INTO procedures_metadata.procedures_params VALUES",
    data_params
)
print(f" Загружено {len(data_params)} записей в procedures_params")


# ============================================
# 7. Проверка и статистика
# ============================================
print("\n=== СТАТИСТИКА ===")

result = client.execute("""
    SELECT 
        (SELECT COUNT(*) FROM procedures_metadata.procedures_params) as params_count,
        (SELECT COUNT(DISTINCT proc_name) FROM procedures_metadata.procedures_params) as procs_count,
        (SELECT COUNT(*) FROM procedures_metadata.procedures_full_text) as full_text_count
""")
params_count, procs_count, full_text_count = result[0]

print(f" Параметров в ClickHouse: {params_count}")
print(f" Уникальных процедур: {procs_count}")
print(f" Процедур с полным текстом: {full_text_count}")
# Выбираем только DMT_процедуры
dmt_procedures = [p for p in unique_procs if p.startswith('DMT_')]
print(f"Найдено DMT-процедур: {len(dmt_procedures)}")
# ============================================
# 8. Пример: посмотрим первые 5 процедур
# ============================================
print("\n=== ПРИМЕР ПРОЦЕДУР (первые 5) ===")

result = client.execute("""
    SELECT DISTINCT proc_name 
    FROM procedures_metadata.procedures_params 
    ORDER BY proc_name 
    LIMIT 5
""")

for (proc_name,) in result:
    # Получаем параметры
    params = client.execute(f"""
        SELECT param_name, param_type, param_order
        FROM procedures_metadata.procedures_params
        WHERE proc_name = '{proc_name}'
        ORDER BY param_order
    """)
    
    print(f"\n {proc_name}")
    for param in params:
        print(f"     - {param[0]} ({param[1]})")

# ============================================
# 9. Закрытие соединения
# ============================================
client.disconnect()
print("\n Готово! Данные сохранены в ClickHouse")

=== Шаг 1: Подключение к MSSQL и выгрузка процедур ===

Подключение к MSSQL: db22.vra.local/mobile_Makeevka
Выгружаем параметры процедур...


/tmp/ipykernel_303/2406638961.py:56: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_params = pd.read_sql(query_params, conn)


 Загружено 8410 записей (процедуры + параметры)

Найдено уникальных процедур: 1626
 Старые данные удалены

--- Загрузка полных текстов всех процедур ---
  Загружено 100 из 1626 процедур
  Загружено 200 из 1626 процедур
  Загружено 300 из 1626 процедур
  Загружено 400 из 1626 процедур
  Загружено 500 из 1626 процедур
  Загружено 600 из 1626 процедур
  Загружено 700 из 1626 процедур
  Загружено 800 из 1626 процедур
  Загружено 900 из 1626 процедур
  Загружено 1000 из 1626 процедур
  Загружено 1100 из 1626 процедур
  Загружено 1200 из 1626 процедур
  Загружено 1300 из 1626 процедур
  Загружено 1400 из 1626 процедур
  Загружено 1500 из 1626 процедур
  Загружено 1600 из 1626 процедур
 Загружено определений: 1626
  Непустых определений: 1626
 Подключение к MSSQL закрыто

=== Шаг 2: Сохранение в ClickHouse ===

 Таблицы в ClickHouse созданы

Загружаем полные тексты в ClickHouse...
 Загружено 1626 записей в procedures_full_text
  Записей в ClickHouse: 1626

Загружаем параметры процедур...
 Заг

In [12]:
# Выполни в Jupyter
result = client.execute("""
    SELECT DISTINCT proc_name 
    FROM procedures_metadata.procedures_full_text 
    WHERE proc_name LIKE 'DMT_%'
    ORDER BY proc_name
    LIMIT 20
""")

print(" Примеры DMT-процедур:")
for row in result:
    print(f"   - {row[0]}")

 Примеры DMT-процедур:
   - DMT_AgentWorkDay
   - DMT_Check_Order
   - DMT_ClearDatabase
   - DMT_Clear_Balance
   - DMT_Clear_FacesDiscounts
   - DMT_Clear_FacesPriceList
   - DMT_Clear_SalesHistory
   - DMT_Clear_Stock
   - DMT_Confirm_ActionLog
   - DMT_Confirm_DocStatus
   - DMT_Confirm_Document
   - DMT_Confirm_GPS
   - DMT_Confirm_MerResults
   - DMT_Confirm_Messages
   - DMT_Confirm_RouteResults
   - DMT_Confirm_RouteResultsEx
   - DMT_Confirm_UsersFiles
   - DMT_Del_DocItem
   - DMT_Del_Document
   - DMT_Del_DocumentDebts


In [15]:
# Посмотрим параметры DMT-процедур
result = client.execute("""
    SELECT 
        p.proc_name,
        COUNT(p.param_name) as param_count,
        groupArray(p.param_name) as params
    FROM procedures_metadata.procedures_params p
    WHERE p.proc_name LIKE 'DMT_%'
    GROUP BY p.proc_name
    LIMIT 10
""")

print(" DMT-процедуры и их параметры:")
for row in result:
    proc_name, param_count, params = row
    print(f"\n {proc_name} ({param_count} параметров)")
    print(f"   Параметры: {', '.join(params)}")

 DMT-процедуры и их параметры:

 DMT_Confirm_GPS (5 параметров)
   Параметры: @ExMasterFID, @BeginDate, @EndDate, @Confirm, @OtherFields

 DMT_Set_Balance (5 параметров)
   Параметры: @clientIDD, @bDate, @bValue, @Limit, @repIDD

 DMT_set_Unit (3 параметров)
   Параметры: @UnitId, @Name, @exID

 DMT_Set_NodesEx (9 параметров)
   Параметры: @NodeId, @NodeExid, @NodeAttrValueID, @NodeAttrValueExId, @NodeName, @NodeFullName, @ActiveFlag, @NodeSort, @OtherFields

 DMT_Import (2 параметров)
   Параметры: @Procname, @TableName

 DMT_AgentWorkDay (3 параметров)
   Параметры: @bDate, @eDate, @FieldsList

 DMT_Get_Stocks (7 параметров)
   Параметры: @OwnerDistId, @OwnerExId, @StockId, @StockExId, @FieldsList, @OtherFields, @ItemsTableExists

 DMT_Get_RoutesEx (5 параметров)
   Параметры: @Option, @BeginDate, @EndDate, @MasterExId, @OtherFields

 DMT_set_AgentEx (8 параметров)
   Параметры: @exid, @activeFlag, @name, @ShortName, @prefix, @StoreIDD, @PersonIDD, @OtherFields

 DMT_Get_Messages (5 

In [16]:
result = client.execute("SELECT COUNT(*) FROM procedures_metadata.procedures_full_text")
print(f"Записей в full_text: {result[0][0]}")

Записей в full_text: 1626


In [1]:
from pathlib import Path
from datetime import datetime, timedelta
import pandas as pd

# Путь к смонтированной папке
ftp_base = Path("/home/jovyan/work/ftp_samples")

# Параметры
days_back = 7
cutoff_date = datetime.now() - timedelta(days=days_back)

total_files = 0
files_info = []

for folder in ftp_base.iterdir():
    if not folder.is_dir():
        continue
    
    try:
        date_str = folder.name.split('_')[0]
        folder_date = datetime.strptime(date_str, '%Y-%m-%d')
        
        if folder_date < cutoff_date:
            continue
        
        for file_path in folder.glob("DMT_*.txt"):
            total_files += 1
            files_info.append({
                'folder': folder.name,
                'file': file_path.name,
                'size': file_path.stat().st_size,
                'date': folder_date
            })
            print(f" {folder.name}/{file_path.name} ({file_path.stat().st_size} bytes)")
            
    except (ValueError, IndexError):
        print(f" Пропускаем: {folder.name}")

print(f"\n Найдено файлов за последние {days_back} дней: {total_files}")

# Покажем структуру
print("\n Список уникальных файлов (типы):")
unique_files = set([f['file'] for f in files_info])
for f in sorted(unique_files):
    print(f"   - {f}")

 2026-03-29_06-20-16-640/DMT_Del_Document.txt (0 bytes)
 2026-03-29_06-20-16-640/DMT_set_ItemTypes.txt (370 bytes)
 2026-03-29_06-20-16-640/DMT_Set_PriceLists.txt (481 bytes)
 2026-03-29_06-20-16-640/DMT_Set_StoreEx.txt (144 bytes)
 2026-03-29_06-20-16-640/DMT_set_ItemsEx.txt (16107 bytes)
 2026-03-29_06-20-16-640/DMT_Set_FirmEx.txt (287 bytes)
 2026-03-29_06-20-16-640/DMT_Set_DocItemEx.txt (955826 bytes)
 2026-03-29_06-20-16-640/DMT_Set_DocumentEx.txt (702885 bytes)
 2026-03-29_06-20-16-640/DMT_set_ItemGroups.txt (48 bytes)
 2026-03-29_06-20-16-640/DMT_Set_AgentEx.txt (3979 bytes)
 2026-03-29_06-20-16-640/DMT_Set_ClientEx.txt (342986 bytes)
 2026-03-29_06-20-16-640/DMT_Confirm_Document.txt (95800 bytes)
 2026-03-29_06-20-16-640/DMT_Set_StockEx.txt (2153 bytes)
 2026-03-29_06-20-16-640/DMT_Set_DocTypes.txt (26 bytes)
 2026-03-29_06-20-16-640/DMT_set_Prices.txt (11635 bytes)
 2026-03-29_06-20-16-640/DMT_Set_PaymentType.txt (107 bytes)
 2026-04-01_06-20-16-273/DMT_Del_Document.txt (0 byt

In [7]:
def read_dmt_file(file_path):
    """Читает DMT-файл с правильной кодировкой"""
    encodings = ['windows-1251', 'cp1251', 'utf-8']
    
    for enc in encodings:
        try:
            with open(file_path, 'r', encoding=enc) as f:
                content = f.read()
                return content, enc
        except UnicodeDecodeError:
            continue
    
    # Если ничего не подошло
    with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
        return f.read(), 'utf-8 (with errors)'

# Тестируем на одном файле
content, enc = read_dmt_file(sample_file)
print(f"Кодировка: {enc}")
print(f"Длина содержимого: {len(content)} символов")

Кодировка: windows-1251
Длина содержимого: 472 символов


In [8]:
from datetime import datetime, timedelta

ftp_base = Path("/home/jovyan/work/ftp_samples")
days_back = 7
cutoff_date = datetime.now() - timedelta(days=days_back)

files_data = []

for folder in ftp_base.iterdir():
    if not folder.is_dir():
        continue
    
    try:
        date_str = folder.name.split('_')[0]
        folder_date = datetime.strptime(date_str, '%Y-%m-%d')
        
        if folder_date < cutoff_date:
            continue
        
        for file_path in folder.glob("DMT_*.txt"):
            content, enc = read_dmt_file(file_path)
            files_data.append({
                'folder': folder.name,
                'file': file_path.name,
                'content': content[:1000],  # сохраняем первые 1000 символов
                'size': len(content),
                'encoding': enc
            })
            print(f" {folder.name}/{file_path.name} ({len(content)} символов, {enc})")
            
    except (ValueError, IndexError):
        print(f" Пропускаем: {folder.name}")

print(f"\n Всего обработано файлов: {len(files_data)}")

 2026-03-29_06-20-16-640/DMT_Del_Document.txt (0 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_set_ItemTypes.txt (363 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_PriceLists.txt (472 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_StoreEx.txt (142 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_set_ItemsEx.txt (16043 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_FirmEx.txt (286 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_DocItemEx.txt (945567 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_DocumentEx.txt (700490 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_set_ItemGroups.txt (47 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_AgentEx.txt (3938 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_ClientEx.txt (341875 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Confirm_Document.txt (93405 символов, windows-1251)
 2026-03-29_06-20-16-640/DMT_Set_StockEx.txt (2127 символов, windows-1251)
 2026-0

In [10]:
# Найдём файл DMT_Set_PriceLists.txt
for item in files_data:
    if 'PriceLists' in item['file']:
        print(f"\n {item['file']} из {item['folder']}:")
        print("-" * 50)
        print(item['content'][:1000])
        break


 DMT_Set_PriceLists.txt из 2026-03-29_06-20-16-640:
--------------------------------------------------
d3f97fe3-9cde-11ef-9ffc-00155d0b0405|СПЦ ВкусМаркет|1
6279e01d-b399-11ee-9fde-00155d0b0405|СПЦ Наместник|1
507bde35-849f-11f0-a018-00155d0b0405|СПЦ Деликатес|1
a7dca030-b396-11ee-9fde-00155d0b0405|БОЦ|1
13861057-b39a-11ee-9fde-00155d0b0405|СПЦ Трамвай|1
88d608f5-b398-11ee-9fde-00155d0b0405|СПЦ Вектор|1
359c9e1d-3c79-11f0-a010-00155d0b0405|СПЦ Сигма Парус|1
f7d967e9-f1ed-11f0-80ec-00155d0beb01|СПЦ Галактика (пекарня)|1
ec482e64-b398-11ee-9fde-00155d0b0405|СПЦ Молоко|1



In [12]:
import pandas as pd
from io import StringIO

# Найдём файл с ценами
for item in files_data:
    if 'set_Prices' in item['file'] and 'DMT_set_Prices.txt' in item['file']:
        print(f"Анализируем: {item['file']}")
        
        # Пробуем прочитать как CSV с разделителем |
        try:
            df = pd.read_csv(StringIO(item['content']), sep='|', header=None)
            print(f"Колонок: {df.shape[1]}")
            print(f"Строк: {df.shape[0]}")
            print("\nПервые 3 строки:")
            print(df.head(3))
        except Exception as e:
            print(f"Ошибка: {e}")
        break

Анализируем: DMT_set_Prices.txt
Колонок: 5
Строк: 12

Первые 3 строки:
                                      0                                     1  \
0  0232d887-ec65-11f0-80ec-00155d0beb01  a7dca030-b396-11ee-9fde-00155d0b0405   
1  a849bf41-0263-11f1-80f5-00155d0beb01  a7dca030-b396-11ee-9fde-00155d0b0405   
2  05eca33a-0264-11f1-80f5-00155d0beb01  a7dca030-b396-11ee-9fde-00155d0b0405   

        2  3   4  
0  588.58  1 NaN  
1  185.62  1 NaN  
2  195.08  1 NaN  


In [13]:
from clickhouse_driver import Client

client = Client(host='my_clickhouse', port=9000, user='default', password='')

# Получаем DMT-процедуры из ClickHouse
result = client.execute("""
    SELECT DISTINCT proc_name 
    FROM procedures_metadata.procedures_full_text 
    WHERE proc_name LIKE 'DMT_%'
    ORDER BY proc_name
""")

procedures = [row[0] for row in result]
print(f" DMT-процедур в ClickHouse: {len(procedures)}")

# Сопоставляем файлы с процедурами
file_types = sorted(set([f['file'] for f in files_data]))
print("\n Сопоставление:")
for file_type in file_types:
    base_name = file_type.replace('.txt', '')
    # Ищем точное совпадение
    if base_name in procedures:
        print(f"    {file_type} → {base_name}")
    else:
        # Ищем частичное совпадение
        matches = [p for p in procedures if base_name.lower() in p.lower() or p.lower() in base_name.lower()]
        if matches:
            print(f"    {file_type} → возможно: {matches[0]}")
        else:
            print(f"    {file_type} → нет соответствия")

 DMT-процедур в ClickHouse: 373

 Сопоставление:
    DMT_Confirm_Document.txt → DMT_Confirm_Document
    DMT_Del_Document.txt → DMT_Del_Document
    DMT_Set_AgentEx.txt → возможно: DMT_Set_AgentEx_TempData_Add
    DMT_Set_ClientEx.txt → DMT_Set_ClientEx
    DMT_Set_DocItemEx.txt → DMT_Set_DocItemEx
    DMT_Set_DocTypes.txt → DMT_Set_DocTypes
    DMT_Set_DocumentEx.txt → DMT_Set_DocumentEx
    DMT_Set_FirmEx.txt → DMT_Set_FirmEx
    DMT_Set_PaymentType.txt → DMT_Set_PaymentType
    DMT_Set_PriceLists.txt → DMT_Set_PriceLists
    DMT_Set_StockEx.txt → DMT_Set_StockEx
    DMT_Set_StoreEx.txt → возможно: DMT_set_Store
    DMT_set_ItemGroups.txt → DMT_set_ItemGroups
    DMT_set_ItemTypes.txt → DMT_set_ItemTypes
    DMT_set_ItemsEx.txt → возможно: DMT_Set_Items
    DMT_set_Prices.txt → DMT_set_Prices


In [14]:
# Пример: файл DMT_Set_PriceLists.txt должен вызывать процедуру DMT_Set_PriceLists
# Посмотрим содержимое файла
for item in files_data:
    if 'PriceLists' in item['file']:
        print(f"\n {item['file']} — данные для процедуры:")
        lines = item['content'].strip().split('\n')
        print(f"Количество записей: {len(lines)}")
        print("Пример записи:")
        print(lines[0] if lines else "пусто")
        break


 DMT_Set_PriceLists.txt — данные для процедуры:
Количество записей: 9
Пример записи:
d3f97fe3-9cde-11ef-9ffc-00155d0b0405|СПЦ ВкусМаркет|1


In [15]:
from clickhouse_driver import Client

client = Client(host='my_clickhouse', port=9000, user='default', password='')

# Таблица маппинга файлов (один раз, для всех регионов)
client.execute("""
    CREATE TABLE IF NOT EXISTS procedures_metadata.file_mapping (
        file_name String,           -- DMT_Set_PriceLists.txt
        proc_name String,           -- DMT_Set_PriceLists
        column_mapping String,      -- JSON: {"columns": [0,1,2], "params": ["@exid", "@name", "@activeflag"]}
        delimiter String DEFAULT '|',
        created_at DateTime DEFAULT now()
    ) ENGINE = MergeTree()
    ORDER BY file_name
""")

print(" Таблица маппинга создана")

 Таблица маппинга создана


In [18]:
from clickhouse_driver import Client

client = Client(host='my_clickhouse', port=9000, user='default', password='')

# Таблица маппинга файлов
client.execute("""
    CREATE TABLE IF NOT EXISTS procedures_metadata.file_mapping (
        file_name String,
        proc_name String,
        column_mapping String,
        delimiter String DEFAULT '|',
        created_at DateTime DEFAULT now()
    ) ENGINE = MergeTree()
    ORDER BY file_name
""")

print(" Таблица маппинга создана")

# Очищаем старые данные
client.execute("TRUNCATE TABLE procedures_metadata.file_mapping")

# Маппинг для известных файлов (вставляем одной командой)
data = [
    ('DMT_Set_PriceLists.txt', 'DMT_Set_PriceLists', '{"columns": [0,1,2], "params": ["@exid", "@name", "@activeflag"]}'),
    ('DMT_set_Prices.txt', 'DMT_set_Prices', '{"columns": [0,1,2], "params": ["@product_guid", "@price_list_guid", "@price"]}'),
    ('DMT_set_ItemsEx.txt', 'DMT_set_ItemsEx', '{"columns": [0,1,2,3,4,5,6,7,8,9,10], "params": ["@guid", "@product_guid", "@name", "@full_name", "@qty", "@price", "@vat", "@discount", "@active", "@flags", "@price_list_guid"]}'),
    ('DMT_Set_ClientEx.txt', 'DMT_Set_ClientEx', '{"columns": [0,1,2], "params": ["@exid", "@name", "@activeflag"]}'),
    ('DMT_Set_AgentEx.txt', 'DMT_Set_AgentEx', '{"columns": [0,1,2,3,4,5,6,7], "params": ["@exid", "@activeFlag", "@name", "@ShortName", "@prefix", "@StoreIDD", "@PersonIDD", "@OtherFields"]}'),
    ('DMT_Set_StockEx.txt', 'DMT_Set_StockEx', '{"columns": [0,1,2,3,4], "params": ["@stock_id", "@product_id", "@quantity", "@price", "@date"]}'),
    ('DMT_Confirm_Document.txt', 'DMT_Confirm_Document', '{"columns": [0,1], "params": ["@doc_id", "@confirm_flag"]}'),
    ('DMT_Set_DocumentEx.txt', 'DMT_Set_DocumentEx', '{"columns": [0,1,2,3,4,5,6], "params": ["@doc_id", "@doc_date", "@client_id", "@sum", "@status", "@comment", "@user_id"]}'),
]

# Вставляем все записи одним запросом
client.execute(
    "INSERT INTO procedures_metadata.file_mapping (file_name, proc_name, column_mapping) VALUES",
    data
)

print(f" Добавлено {len(data)} записей")

# Проверяем
result = client.execute("SELECT COUNT(*) FROM procedures_metadata.file_mapping")
print(f" Всего записей в маппинге: {result[0][0]}")

 Таблица маппинга создана
 Добавлено 8 записей
 Всего записей в маппинге: 8


In [2]:
from pathlib import Path
from datetime import datetime
import csv

# Маппинг (добавь остальные)
KNOWN_STRUCTURES = {
    "DMT_Set_PriceLists.txt": {
        "columns": 3,
        "column_types": ["guid", "string", "bool"],
        "delimiter": "|"
    },
    "DMT_set_Prices.txt": {
        "columns": 5,
        "column_types": ["guid", "guid", "number", "number", "string"],
        "delimiter": "|"
    },
}

def validate_file(file_path, structure):
    """Проверяет файл, возвращает список ошибок"""
    errors = []
    try:
        with open(file_path, 'r', encoding='windows-1251') as f:
            lines = f.readlines()
    except:
        return ["Cannot read file"]
    
    if not lines:
        errors.append("Empty file")
        return errors
    
    delimiter = structure["delimiter"]
    expected_cols = structure["columns"]
    
    for i, line in enumerate(lines, 1):
        if not line.strip():
            continue
        cols = line.strip().split(delimiter)
        if len(cols) != expected_cols:
            errors.append(f"Line {i}: {len(cols)} columns, expected {expected_cols}")
    
    return errors

# Сканируем файлы
ftp_base = Path("/home/jovyan/work/ftp_local")
results = []

for region in ftp_base.iterdir():
    if not region.is_dir():
        continue
    for folder in region.iterdir():
        if not folder.is_dir():
            continue
        for file_path in folder.glob("DMT_*.txt"):
            if file_path.name in KNOWN_STRUCTURES:
                errors = validate_file(file_path, KNOWN_STRUCTURES[file_path.name])
                results.append({
                    "region": region.name,
                    "folder": folder.name,
                    "file": file_path.name,
                    "errors": len(errors),
                    "error_list": "; ".join(errors[:3])
                })
            else:
                results.append({
                    "region": region.name,
                    "folder": folder.name,
                    "file": file_path.name,
                    "errors": -1,
                    "error_list": "UNKNOWN FILE TYPE"
                })

# Статистика
total = len(results)
ok_files = sum(1 for r in results if r["errors"] == 0)
error_files = sum(1 for r in results if r["errors"] > 0)
unknown_files = sum(1 for r in results if r["errors"] == -1)

print(f" СТАТИСТИКА:")
print(f"   Всего файлов: {total}")
print(f"    OK: {ok_files}")
print(f"    Ошибки: {error_files}")
print(f"    Неизвестные: {unknown_files}")

# Покажем первые 10 ошибок
print(f"\n ПЕРВЫЕ 10 ФАЙЛОВ С ОШИБКАМИ:")
error_results = [r for r in results if r["errors"] > 0]
for r in error_results[:10]:
    print(f"   {r['region']}/{r['folder']}/{r['file']}: {r['error_list']}")

 СТАТИСТИКА:
   Всего файлов: 2570
    OK: 161
    Ошибки: 3
    Неизвестные: 2406

 ПЕРВЫЕ 10 ФАЙЛОВ С ОШИБКАМИ:
   kursk/2026-03-27/DMT_Set_PriceLists.txt: Line 1: 4 columns, expected 3
   kursk/2026-03-30/DMT_Set_PriceLists.txt: Line 1: 4 columns, expected 3
   kursk/2026-04-02/DMT_Set_PriceLists.txt: Line 1: 4 columns, expected 3


In [3]:
import shutil
from pathlib import Path

# Создаём папки для ноутбуков
notebooks_dir = Path("/home/jovyan/work/projects/ftp_agent/notebooks")
notebooks_dir.mkdir(parents=True, exist_ok=True)

# Копируем текущий ноутбук (если он сохранён)
# Сначала сохрани текущий ноутбук как "file_validator.ipynb"
# File → Save As → /home/jovyan/work/projects/ftp_agent/notebooks/file_validator.ipynb

print(f" Ноутбук сохранён в: {notebooks_dir}")

 Ноутбук сохранён в: /home/jovyan/work/projects/ftp_agent/notebooks


In [21]:
# Базовый путь (должен быть смонтирован)
ftp_root = Path("/home/jovyan/work/ftp_root")

# Все регионы
regions = [p.name for p in ftp_root.iterdir() if p.is_dir()]
print(f" Найдено регионов: {len(regions)}")

# Параметры
days_back = 7
cutoff_date = datetime.now() - timedelta(days=days_back)

total_files = 0

for region in regions:
    backup_path = ftp_root / region / "BackUp_Import"
    if not backup_path.exists():
        print(f" Нет папки BackUp_Import в {region}")
        continue
    
    print(f"\n Регион: {region}")
    
    for folder in backup_path.iterdir():
        if not folder.is_dir():
            continue
        
        try:
            date_str = folder.name.split('_')[0]
            folder_date = datetime.strptime(date_str, '%Y-%m-%d')
            if folder_date < cutoff_date:
                continue
            
            print(f"    {folder.name}")
            
            for file_path in folder.glob("DMT_*.txt"):
                total_files += 1
                process_file(file_path, region, date_str)
                
        except (ValueError, IndexError):
            print(f"    Пропускаем папку: {folder.name}")

print(f"\n Всего обработано файлов: {total_files}")

 Найдено регионов: 1

 Регион: makeevka

 Всего обработано файлов: 0


In [22]:
# Посмотрим, сколько файлов обработано
result = client.execute("""
    SELECT 
        status,
        COUNT(*) as count
    FROM procedures_metadata.processed_files
    GROUP BY status
""")

print("\n СТАТИСТИКА ОБРАБОТКИ:")
for row in result:
    print(f"   {row[0]}: {row[1]}")

# Посмотрим последние обработанные файлы
result = client.execute("""
    SELECT region, folder_date, file_name, status, processed_at
    FROM procedures_metadata.processed_files
    ORDER BY processed_at DESC
    LIMIT 10
""")

print("\n ПОСЛЕДНИЕ 10 ФАЙЛОВ:")
for row in result:
    print(f"   {row[0]}/{row[1]}/{row[2]} → {row[3]} ({row[4]})")


 СТАТИСТИКА ОБРАБОТКИ:

 ПОСЛЕДНИЕ 10 ФАЙЛОВ:


In [4]:
!pip install chardet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 990.4 kB/s eta 0:00:000:0100:01m


In [5]:
import chardet

sample_file = ftp_base / "2026-04-02_06-20-15-990" / "DMT_Set_PriceLists.txt"

# Определяем кодировку
with open(sample_file, 'rb') as f:
    raw_data = f.read()
    result = chardet.detect(raw_data)
    print(f"Определённая кодировка: {result}")

Определённая кодировка: {'encoding': 'Windows-1251', 'confidence': 0.07251236650036624, 'language': 'be', 'mime_type': 'text/plain'}


In [18]:
from clickhouse_driver import Client

client = Client(host='my_clickhouse', port=9000, user='default', password='')

# Создаём таблицу для хранения эмбеддингов файлов
client.execute("""
    CREATE TABLE IF NOT EXISTS procedures_metadata.file_embeddings (
        file_name String,
        content String,
        embedding Array(Float32),
        processed_at DateTime
    ) ENGINE = MergeTree()
    ORDER BY processed_at
""")

print(" Таблица file_embeddings создана в ClickHouse")

# Проверяем
result = client.execute("SHOW TABLES FROM procedures_metadata")
print(f"\n Таблицы в procedures_metadata:")
for row in result:
    print(f"   - {row[0]}")


from sentence_transformers import SentenceTransformer

# Загружаем модель
model = SentenceTransformer('all-MiniLM-L6-v2')

# Читаем файл
with open(file_path, 'r') as f:
    content = f.read()

# Создаём эмбеддинг
embedding = model.encode(content)

# Сохраняем в ClickHouse
client.execute("""
    INSERT INTO procedures_metadata.file_embeddings (file_name, content, embedding, processed_at)
    VALUES (%s, %s, %s, now())
""", (file_name, content[:5000], embedding.tolist()))



import os
from pathlib import Path
from datetime import datetime, timedelta

# Базовый путь
ftp_root = Path("//vra.local/Root/FTP/makeevka/BackUp_Import")

# Нас интересуют только папки за последние N дней
lookback_days = 7  # проверяем только последние 7 дней
cutoff_date = datetime.now() - timedelta(days=lookback_days)

# Фильтруем папки по дате в имени
for folder in ftp_root.iterdir():
    if not folder.is_dir():
        continue
    
    # Имя папки: 2026-04-02_06-20-15-990
    try:
        date_str = folder.name.split('_')[0]  # '2026-04-02'
        folder_date = datetime.strptime(date_str, '%Y-%m-%d')
        
        if folder_date >= cutoff_date:
            print(f" Обрабатываем: {folder.name}")
            # Здесь читаем файлы внутри папки
    except (ValueError, IndexError):
        print(f" Пропускаем: {folder.name} (не распознана дата)")

 Таблица file_embeddings создана в ClickHouse

Загружаем модель sentence-transformers...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


 Модель загружена
 Файл не найден: /home/jovyan/work/projects/ftp_agent/data/test_files/DMT_set_PriceLists.txt
